# Análisis histórico de los datos (Puntos 7 y 8)
Este notebook sigue el trabajo de `01_limpieza_y_variables.ipynb`. Primero se cargan los datos y se repite el merge y la variable `medalla` para tener el mismo dataframe.

In [2]:
import pandas as pd
import numpy as np

df_atletas = pd.read_csv('../data/athlete_events.csv')
df_regiones = pd.read_csv('../data/noc_regions.csv')

df_atletas['IMC'] = df_atletas['Weight'] / ((df_atletas['Height'] / 100) ** 2)
df_atletas = pd.merge(df_atletas, df_regiones, on='NOC', how='left')
df_atletas['medalla'] = df_atletas['Medal'].notna().astype(int)

print(df_atletas.shape)

ModuleNotFoundError: No module named 'pandas'

### 7. ¿Los datos son representativos de toda la historia olímpica?

In [ ]:
# se divide la historia en epocas para comparar
df_atletas['epoca'] = pd.cut(df_atletas['Year'],
                             bins=[1895, 1945, 1975, 2000, 2016],
                             labels=['1896-1945', '1946-1975', '1976-2000', '2001-2016'])

print("Registros por época:")
print(df_atletas['epoca'].value_counts().sort_index())

print("\nPorcentaje de registros por temporada:")
print((df_atletas['Season'].value_counts(normalize=True) * 100).round(1))

print("\nPorcentaje de registros por sexo:")
print((df_atletas['Sex'].value_counts(normalize=True) * 100).round(1))

In [3]:
# participacion de hombres y mujeres en cada epoca (en %)
tabla_sexo_epoca = pd.crosstab(df_atletas['epoca'], df_atletas['Sex'], normalize='index') * 100
display(tabla_sexo_epoca.round(1))

# verano vs invierno en cada epoca (en %)
tabla_season_epoca = pd.crosstab(df_atletas['epoca'], df_atletas['Season'], normalize='index') * 100
display(tabla_season_epoca.round(1))

tabla_sexo_epoca.round(1).to_csv('../outputs/tablas/sexo_por_epoca.csv')
tabla_season_epoca.round(1).to_csv('../outputs/tablas/temporada_por_epoca.csv')

NameError: name 'pd' is not defined

### Análisis de representatividad

* **Por época:** los registros no están repartidos igual en el tiempo. La mitad de los datos son de 1988 en adelante y solo una cuarta parte es de antes de 1960, entonces las ediciones recientes pesan mucho más que las primeras. Esto pasa porque los Juegos crecieron mucho con los años, con más países, más deportes y más pruebas.
* **Por temporada:** cerca del 82% de los registros son de los Juegos de Verano y solo el 18% de Invierno. Los de Invierno empezaron en 1924 y tienen muchas menos pruebas, así que cualquier conclusión general va a estar dominada por lo que pasa en Verano.
* **Por sexo:** alrededor del 72.5% de los registros son de hombres. En las primeras épocas las mujeres casi no aparecen y su participación va subiendo con el tiempo, por lo que en las ediciones recientes la diferencia es mucho menor.
* **Conclusión:** los datos sí cubren toda la historia (1896 a 2016), pero no la representan de forma equilibrada. Si se sacan promedios o proporciones con toda la base, el resultado se parece más a los Juegos modernos de Verano con mayoría de hombres. Por eso es mejor comparar dentro de cada época, temporada o sexo en vez de mezclar todo.

### 8. Datos faltantes y su cambio antes y después de 1960

In [4]:
faltantes = pd.DataFrame({
    'faltantes': df_atletas.isna().sum(),
    'porcentaje': (df_atletas.isna().mean() * 100).round(2)
})
display(faltantes[faltantes['faltantes'] > 0].sort_values('porcentaje', ascending=False))

NameError: name 'pd' is not defined

In [5]:
# antes y despues de 1960
df_atletas['periodo'] = np.where(df_atletas['Year'] < 1960, 'Antes de 1960', '1960 en adelante')

cols = ['Age', 'Height', 'Weight', 'IMC']
tabla_faltantes_periodo = (df_atletas.groupby('periodo')[cols]
                           .apply(lambda g: g.isna().mean() * 100)
                           .round(1))
display(tabla_faltantes_periodo)

tabla_faltantes_periodo.to_csv('../outputs/tablas/faltantes_antes_despues_1960.csv')

NameError: name 'np' is not defined

### Análisis de datos faltantes

* **¿Qué variables tienen faltantes?** `Age` (9,474 filas, cerca del 3.5%), `Height` (60,171, cerca del 22%), `Weight` (62,875, cerca del 23%), `IMC` (porque sale de estatura y peso, si falta uno de los dos queda vacío), `Medal` (231,333), `region` (370 por los NOC que no cruzaron en el merge) y `notes`.
* **¿Qué significan?** No todos los faltantes son iguales. En `Medal` el vacío significa que el atleta no ganó medalla, no es un error. En `notes` casi siempre está vacío porque solo se usa para aclarar casos especiales. En cambio, en `Age`, `Height` y `Weight` sí es información que no se registró, sobre todo en las primeras ediciones donde no se llevaban esos datos de los atletas.
* **¿Cambia según la época?** Sí. En la tabla se ve que antes de 1960 el porcentaje de estatura y peso faltantes es mucho más alto que de 1960 en adelante, y lo mismo pasa con la edad aunque en menor medida. Esto tiene sentido porque los registros antiguos eran menos completos.
* **Implicación:** cualquier análisis con estatura, peso o IMC va a estar basado sobre todo en los Juegos modernos, porque los antiguos casi no tienen esos datos.